In [1]:
# use "env" environment 

from src.create_graphs import create_graph_list
from src.load_data import load_data
from src.VSA_conversion import VSA_conversion
from src.embeddings import getEmbedding
from models.graphcnnVSA_Binding_FULL import GraphCNN
import torch
from xgboost import XGBRegressor
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors,rdMolDescriptors

from rdkit import Chem,DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen

### Importing the required library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MolStandardize
import joblib
from src import utilities




# Traditional Feature extraction

In [2]:
train_set=pd.read_csv('final_data/final_unique_train_fixed.csv')
test_set=pd.read_csv('final_data/final_unique_test.csv')

train_smiles_list=train_set[['smiles_canon']]
test_smiles_list=test_set[['smiles_canon']]


### Generate 4 descriptors ....
df4_train=utilities.generate4(train_set.smiles_canon)
df4_test=utilities.generate4(test_set.smiles_canon)
### Generate 17 descriptors ....
df17_train=utilities.generate17(train_set.smiles_canon)
df17_test=utilities.generate17(test_set.smiles_canon)
### Generate 123 descriptors ....'''
df123_train=utilities.generate123(train_set.smiles_canon)
df123_test=utilities.generate123(test_set.smiles_canon)
### Generate 38 feature engineered based on the structure of the smiles ....
df38_train=utilities.generate_features38(train_set.smiles_canon)
df38_test=utilities.generate_features38(test_set.smiles_canon)
### Generate 7 funnctional groups
df7_train=utilities.get_functional_groups(train_set.smiles_canon)
df7_test=utilities.get_functional_groups(test_set.smiles_canon)
### Fingerprint 128....
df128_train=utilities.fingerprint(train_set.smiles_canon,2,128)
df128_test=utilities.fingerprint(test_set.smiles_canon,2,128)


## Prof. Ulf proposed data
df96_train=utilities.generate_desc_96(train_set.smiles_canon)
df96_test=utilities.generate_desc_96(test_set.smiles_canon)

## Prof. Ulf + chatgpt suggested data
df193_train=utilities.generate_desc_193(train_set.smiles_canon)
df193_test=utilities.generate_desc_193(test_set.smiles_canon)

In [3]:
df298_train=pd.concat([df123_train, df128_train, df7_train, df38_train], axis=1)
df298_test=pd.concat([df123_test, df128_test, df7_test, df38_test], axis=1)

In [4]:
a = list(df298_test[0])

print(max(a))
print(min(a))

1
0


# GVFA Feature extraction

#### Load data and make graph

#### Make hypervectors and combine  with traditional Features  - CONCATINATION

GVFA feature size - 100, 500, 1000, 2000, 5000, 10000
Traditional feature size = 298
Convert traditional feature set inton torch and do the cobcatination  

In [4]:
HV_Dimentions = [100, 500, 1000, 2000, 5000, 10000]

for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()


    # train_graphs = create_graph_list(train_data)
    # test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0)
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0)

    df_np = df298_train.to_numpy(dtype=np.float32)          # or df298_train.fillna(0).to_numpy(np.float32)
    df_torch_train  = torch.from_numpy(df_np)  

    df_np_ = df298_test.to_numpy(dtype=np.float32)  
    df_torch_test  = torch.from_numpy(df_np_)   
    

    X_train = torch.cat([df_torch_train, train_embeddings_eq1], axis=1)

    # X_train = pd.concat([df_t, train_embeddings_eq1], axis=1)
    X_test = torch.cat([df_torch_test, test_embeddings_eq1], axis=1)

    print(X_train.shape)
    print(X_test.shape)

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )

    xgb.fit(
        X_train, train_labels_eq1,
        eval_set=[(X_test, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(X_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 concatinate GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

KeyboardInterrupt: 

In this experiment, standardize all featutres as ROSS said before concatination

In [ ]:
from sklearn.preprocessing import StandardScaler

HV_Dimentions = [100, 500, 1000, 2000, 5000, 10000]

scaler_298 = StandardScaler()
scaler_298.fit(df298_train.values)   # each column: its own mean/std


for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()
    print(train_data[0].edge_attr)

    # train_graphs = create_graph_list(train_data)
    # test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0)
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0)

    df298_train_scaled = scaler_298.transform(df298_train.values)
    df298_test_scaled  = scaler_298.transform(df298_test.values)


    df_torch_train = torch.from_numpy(df298_train_scaled.astype(np.float32))
    df_torch_test  = torch.from_numpy(df298_test_scaled.astype(np.float32))


    X_train = torch.cat([df_torch_train, train_embeddings_eq1], axis=1)

    # X_train = pd.concat([df_t, train_embeddings_eq1], axis=1)
    X_test = torch.cat([df_torch_test, test_embeddings_eq1], axis=1)

    print(X_train.shape)
    print(X_test.shape)

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )

    xgb.fit(
        X_train, train_labels_eq1,
        eval_set=[(X_test, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(X_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 concatinate GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

tensor([1, 1, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 1, 1, 0, 0, 3, 3, 0, 0,
        3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3])
VSA_conversion 16
W :  torch.Size([16, 100])
g list item shape before :  torch.Size([5, 16])
g list item shape after :  torch.Size([5, 100])
VSA_conversion 16
W :  torch.Size([16, 100])
g list item shape before :  torch.Size([17, 16])
g list item shape after :  torch.Size([17, 100])
Input feature size:  100


NotFittedError: This StandardScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In this experiment, standardize all featutres before concatination

In [5]:
HV_Dimentions = [100, 500, 1000, 2000, 5000, 10000]
from sklearn.preprocessing import StandardScaler, RobustScaler
for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()


    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0)
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0)

    df_np = df298_train.to_numpy(dtype=np.float32)          # or df298_train.fillna(0).to_numpy(np.float32)
    df_torch_train  = torch.from_numpy(df_np)  

    df_np_ = df298_test.to_numpy(dtype=np.float32)  
    df_torch_test  = torch.from_numpy(df_np_)   

    
    def l2_rows(X):
        return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    
    def fit_transform_scaler(X_train, X_test, scaler):
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        return Xtr, Xte, scaler
    
    # sc_df  = RobustScaler(with_centering=True, with_scaling=True)
    sc_hv  = StandardScaler(with_mean=True, with_std=True)
    sc_df  = StandardScaler(with_mean=True, with_std=True)
    X_df_tr_s, X_df_te_s, _ = fit_transform_scaler(df_torch_train, df_torch_test, sc_df)
    HV_tr_s,   HV_te_s,   _ = fit_transform_scaler(train_embeddings_eq1,   test_embeddings_eq1,   sc_hv)

    # print(df_torch_train.shape)
    # print(df_torch_test.shape)

    X_train = torch.cat([torch.from_numpy(X_df_tr_s), torch.from_numpy(HV_tr_s)], axis=1)
    X_test = torch.cat([torch.from_numpy(X_df_te_s), torch.from_numpy(HV_te_s)], axis=1)

    print(X_train.shape)
    print(X_test.shape)

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )

    xgb.fit(
        X_train, train_labels_eq1,
        eval_set=[(X_test, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(X_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 concatinate GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

VSA_conversion 7
W :  torch.Size([7, 100])
VSA_conversion 7
W :  torch.Size([7, 100])
Input feature size:  100
torch.Size([17929, 396])
torch.Size([1282, 396])
                      Model_Name     MAE     MSE    RMSE      R2  \
0  XGB_298 concatinate GVFA(100)  0.4229  0.3321  0.5763  0.9204   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 500])
VSA_conversion 7
W :  torch.Size([7, 500])
Input feature size:  500
torch.Size([17929, 796])
torch.Size([1282, 796])
                      Model_Name     MAE     MSE    RMSE      R2  \
0  XGB_298 concatinate GVFA(500)  0.4173  0.3269  0.5718  0.9216   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 1000])
VSA_conversion 7
W :  torch.Size([7, 1000])
Input feature size:  1000
torch.Size([17929, 1296])
torch.Size([1282, 1296])
                     

In this experiment, when concatinate features, we convert 298 feature set into **same size** of GVFA feature set.

GVFA feature size - 100, 500, 1000, 2000, 5000, 10000
Traditional feature size = 298
Convert traditional feature set inton torch and do the cobcatination

In [6]:
from sklearn.random_projection import GaussianRandomProjection

HV_Dimentions = [5000, 10000]

for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()


    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0)
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0)

    proj = GaussianRandomProjection(
        n_components=HV_Dimention,
        random_state=42  # fix for reproducibility, or vary for multiple seeds
    )

    df298_train_nan = np.nan_to_num(df298_train, nan=0.0, posinf=0.0, neginf=0.0)
    df298_test_nan  = np.nan_to_num(df298_test,  nan=0.0, posinf=0.0, neginf=0.0)

    df298_train_rp = proj.fit_transform(df298_train_nan)   # [N_train, HV_Dimention]
    df298_test_rp  = proj.transform(df298_test_nan) 


    # df_np = df298_train_rp.to_numpy(dtype=np.float32)          # or df298_train.fillna(0).to_numpy(np.float32)
    df_torch_train  = torch.from_numpy(df298_train_rp)  

    # df_np_ = df298_test_rp.to_numpy(dtype=np.float32)  
    df_torch_test  = torch.from_numpy(df298_test_rp)   

    # print(df_torch_train.shape)
    # print(df_torch_test.shape)

    X_train = torch.cat([df_torch_train, train_embeddings_eq1], axis=1)

    # X_train = pd.concat([df_t, train_embeddings_eq1], axis=1)
    X_test = torch.cat([df_torch_test, test_embeddings_eq1], axis=1)

    print(X_train.shape)
    print(X_test.shape)

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )
    
    xgb.fit(
        X_train, train_labels_eq1,
        eval_set=[(test_embeddings_eq1, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(X_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 concatinate GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

VSA_conversion 7
W :  torch.Size([7, 5000])
VSA_conversion 7
W :  torch.Size([7, 5000])
Input feature size:  5000
torch.Size([17929, 10000])
torch.Size([1282, 10000])


KeyboardInterrupt: 

Here we concatinate 298 features with HV from GVFA. before concatination, we apply PCA for GVFA feature set to reduce dimention to 100

In [ ]:
from sklearn.random_projection import GaussianRandomProjection
from sklearn.decomposition import PCA

HV_Dimentions = [1000, 2000, 5000, 10000]

for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()


    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0).detach().cpu().numpy()
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0).detach().cpu().numpy()

    pca_dim = 100  # number of PCA components you want
    pca = PCA(n_components=pca_dim, random_state=42)
    X_train_hv_pca = pca.fit_transform(train_embeddings_eq1)   # (N_train × pca_dim)
    X_test_hv_pca  = pca.transform(test_embeddings_eq1)        # (N_test  × pca_dim)

    # 3) Convert back to torch (if you want)
    train_embeddings_eq1 = torch.from_numpy(X_train_hv_pca).float()
    test_embeddings_eq1  = torch.from_numpy(X_test_hv_pca).float()

    # proj = GaussianRandomProjection(
    #     n_components=HV_Dimention,
    #     random_state=42  # fix for reproducibility, or vary for multiple seeds
    # )

    # df298_train_nan = np.nan_to_num(df298_train, nan=0.0, posinf=0.0, neginf=0.0)
    # df298_test_nan  = np.nan_to_num(df298_test,  nan=0.0, posinf=0.0, neginf=0.0)

    # df298_train_rp = proj.fit_transform(df298_train_nan)   # [N_train, HV_Dimention]
    # df298_test_rp  = proj.transform(df298_test_nan) 


    df_np = df298_train.to_numpy(dtype=np.float32)          # or df298_train.fillna(0).to_numpy(np.float32)
    df_torch_train  = torch.from_numpy(df298_train_rp)  

    df_np_ = df298_test.to_numpy(dtype=np.float32)  
    df_torch_test  = torch.from_numpy(df298_test_rp)   

    X_train = torch.cat([df_torch_train, train_embeddings_eq1], axis=1)

    # X_train = pd.concat([df_t, train_embeddings_eq1], axis=1)
    X_test = torch.cat([df_torch_test, test_embeddings_eq1], axis=1)

    print(X_train.shape)
    print(X_test.shape)

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )

    xgb.fit(
        X_train, train_labels_eq1,
        eval_set=[(test_embeddings_eq1, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(X_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 concatinate GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

VSA_conversion 7
W :  torch.Size([7, 100])
VSA_conversion 7
W :  torch.Size([7, 100])
Input feature size:  100
torch.Size([17929, 5100])
torch.Size([1282, 5100])
                      Model_Name     MAE     MSE   RMSE      R2  \
0  XGB_298 concatinate GVFA(100)  0.5081  0.4382  0.662  0.8949   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 500])
VSA_conversion 7
W :  torch.Size([7, 500])
Input feature size:  500
torch.Size([17929, 5100])
torch.Size([1282, 5100])
                      Model_Name     MAE     MSE    RMSE      R2  \
0  XGB_298 concatinate GVFA(500)  0.4881  0.4235  0.6508  0.8985   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 1000])
VSA_conversion 7
W :  torch.Size([7, 1000])
Input feature size:  1000
torch.Size([17929, 5100])
torch.Size([1282, 5100])


KeyboardInterrupt: 

# Make hypervectors and combine  with traditional Features  - Bundling

In [10]:
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.random_projection import GaussianRandomProjection
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import ElasticNet


def make_clean_pipeline(rp_dim=1000, use_knn=False):
    # Pre-step: replace infs with NaN so imputers can handle them
    def _prep(df):
        df = df.replace([np.inf, -np.inf], np.nan)
        return df

    # Choose imputer
    imputer = KNNImputer(n_neighbors=5, weights="distance") if use_knn else SimpleImputer(strategy="median")

    pipe = Pipeline([
        ("prep",    ("passthrough")),              # placeholder; we'll call _prep manually
        ("impute",  imputer),                      # keep rows, fill NaNs
        ("clipper", RobustScaler(with_centering=True, with_scaling=True)),  # robust to outliers
        ("rp",      GaussianRandomProjection(n_components=rp_dim, random_state=42)),
    ])
    pipe._prep = _prep
    return pipe

In [ ]:
from sklearn.random_projection import GaussianRandomProjection

HV_Dimentions = [100, 500, 1000, 2000, 5000, 10000]

for HV_Dimention in HV_Dimentions:

    train_data, test_data = load_data()


    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)


    num_layers = 5
    delta_eq1 = 1
    equation_eq1 = 10
    graph_pooling_type = 'sum'  # sum, average
    neighbor_pooling_type = 'sum' # sum, average, max
    device = 1  # help='if delta is 1 will be the model with binding, if 0 model will have be without binding (default: 1)'
    device = torch.device('cpu')
    
    train_graphs = create_graph_list(train_data)
    test_graphs = create_graph_list(test_data)
    ts_graph = test_graphs.copy()
    tr_graph = train_graphs.copy()

    test_HVs = VSA_conversion(ts_graph, HV_Dimention)
    train_HVs = VSA_conversion(tr_graph, HV_Dimention)

    model_eq1 = GraphCNN(test_HVs[0].node_features.shape[1], num_layers, delta_eq1, graph_pooling_type, neighbor_pooling_type, device, equation_eq1) #.to(device)
    train_embeddings_eq1, train_labels_eq1 = getEmbedding(model_eq1, device, train_HVs)
    test_embeddings_eq1, test_labels_eq1 = getEmbedding(model_eq1, device, test_HVs)

    train_embeddings_eq1 = train_embeddings_eq1.squeeze(0)
    test_embeddings_eq1 = test_embeddings_eq1.squeeze(0)

    pipe = make_clean_pipeline(rp_dim=HV_Dimention, use_knn=False)

    Xtr_df = pipe._prep(df298_train).astype(np.float32)
    Xte_df = pipe._prep(df298_test).astype(np.float32)

    Xtr_rp = pipe.fit_transform(Xtr_df)   # fit on train
    Xte_rp = pipe.transform(Xte_df)       # transform test


    print(Xtr_rp.shape)
    print(train_embeddings_eq1.shape)

    A = Xtr_rp.astype(np.float32)
    A_GVFA = train_embeddings_eq1.detach().cpu().numpy().astype(np.float32)

    B = Xte_rp.astype(np.float32)
    B_GVFA = test_embeddings_eq1.detach().cpu().numpy().astype(np.float32)

    S_train = A + A_GVFA
    S_test = B + B_GVFA


    S_train /= (np.linalg.norm(S_train, axis=1, keepdims=True) + 1e-8)
    S_test /= (np.linalg.norm(S_test, axis=1, keepdims=True) + 1e-8)


    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=4,
        tree_method="hist"   # fast on CPU; use "gpu_hist" if you have GPU
    )

    xgb.fit(
        S_train, train_labels_eq1,
        eval_set=[(S_test, test_labels_eq1)],
        # early_stopping_rounds=100,
        verbose=False
    )

    pred_xgb = xgb.predict(S_test)
    
    xgb_298=utilities.get_errors1(test_labels_eq1,pred_xgb,f"XGB_298 superposition GVFA({HV_Dimention})")
    xgb_298['Descriptors_Detail']='125 features + 128 fingerprint 7 f_group+38 fe features'
    print(xgb_298)

VSA_conversion 7
W :  torch.Size([7, 100])
VSA_conversion 7
W :  torch.Size([7, 100])
Input feature size:  100
(17929, 100)
torch.Size([17929, 100])
                        Model_Name     MAE     MSE    RMSE      R2  \
0  XGB_298 superposition GVFA(100)  0.6526  0.7252  0.8516  0.8261   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 500])
VSA_conversion 7
W :  torch.Size([7, 500])
Input feature size:  500
(17929, 500)
torch.Size([17929, 500])
                        Model_Name    MAE    MSE    RMSE      R2  \
0  XGB_298 superposition GVFA(500)  0.515  0.475  0.6892  0.8861   

                                  Descriptors_Detail  
0  125 features + 128 fingerprint 7 f_group+38 fe...  
VSA_conversion 7
W :  torch.Size([7, 1000])
VSA_conversion 7
W :  torch.Size([7, 1000])


KeyboardInterrupt: 

Normalize 298 feature set. do columnwise normalization 